# 🏠 House Prices — Stacking & Blending Avanzado

> **Kaggle Competition** | Chalito · LYONSOFT PERU
> Notebook dedicado exclusivamente a técnicas de ensemble de alto nivel.

---
### Posición en el pipeline
```
📓 01_EDA.ipynb                  ✅
📓 02_FeatureEngineering.ipynb   ✅
📓 03_Preprocessing_Final.ipynb  ✅
📓 04_Models.ipynb               ✅  → OOF + modelos entrenados
📓 05_Stacking_Blending.ipynb    ◀ ESTE NOTEBOOK
```

### Flujo interno
```
OOF predictions (6 modelos base)
        │
        ├─ Blending simple (promedio igual)
        ├─ Blending SLSQP (scipy optimize)
        ├─ Blending Differential Evolution (global)
        ├─ Blending por subconjunto óptimo
        ├─ Stacking L1 (meta-learner Ridge/Lasso/ENet)
        ├─ Stacking L2 (meta-meta-learner)
        └─ Ensemble Híbrido (L1 + L2 + Blend combinados)
                │
                ▼
          submission_stacking.csv → objetivo RMSLE < 0.115
```

In [ ]:
# ─── 0. IMPORTACIONES ───────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy.optimize import minimize, differential_evolution
from scipy import stats as _stats
from scipy.stats import skew as _skew, norm as _norm
from itertools import combinations
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error
from matplotlib.patches import Patch
import joblib, os, json, copy

plt.rcParams.update({
    'figure.dpi': 120, 'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa', 'axes.grid': True, 'grid.alpha': 0.35,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.family': 'sans-serif', 'axes.titlesize': 12,
})
COLOR_MAIN    = '#2563EB'
COLOR_ACCENT  = '#F59E0B'
COLOR_DANGER  = '#EF4444'
COLOR_SUCCESS = '#10B981'
COLOR_PURPLE  = '#7C3AED'

pd.set_option('display.float_format', '{:.6f}'.format)

N_FOLDS = 10
KF = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_error(np.asarray(y_true), np.asarray(y_pred)))

print('✅ Setup completo  |  K-Folds:', N_FOLDS)

---
## 1 · Carga de artefactos del Notebook 04

In [ ]:
# ─── 1.1 CARGAR DATASETS, TARGET Y OOF PREDICTIONS ─────────────────────────
X_train_scaled = np.load('X_train_scaled.npy')
X_test_scaled  = np.load('X_test_scaled.npy')
X_train_raw    = np.load('X_train_raw.npy')
X_test_raw     = np.load('X_test_raw.npy')
y              = pd.Series(np.load('y_train_log.npy'), name='y_log')
test_ids       = pd.read_csv('test_ids.csv').squeeze()

# OOF y predicciones de test de los modelos base (generadas en notebook 04)
meta_train = np.load('oof_predictions.npy')       # shape: (ntrain, n_base)
meta_test  = np.load('test_predictions_base.npy') # shape: (ntest,  n_base)

BASE_NAMES = ['Ridge', 'Lasso', 'ElasticNet', 'XGBoost', 'LightGBM', 'GBR']
n_base     = len(BASE_NAMES)

print('─' * 55)
print(f'  X_train_scaled : {X_train_scaled.shape}')
print(f'  X_train_raw    : {X_train_raw.shape}')
print(f'  y              : {y.shape}')
print(f'  meta_train     : {meta_train.shape}  ← OOF de {n_base} modelos base')
print(f'  meta_test      : {meta_test.shape}')
print('─' * 55)

base_scores = {}
print('\n📊 RMSLE OOF — modelos base:')
for i, name in enumerate(BASE_NAMES):
    s = rmsle(y, meta_train[:, i])
    base_scores[name] = s
    print(f'   {name:<15} {s:.5f}')

best_base_name  = min(base_scores, key=base_scores.get)
best_base_score = base_scores[best_base_name]
print(f'\n   Mejor base: {best_base_name} → {best_base_score:.5f}')

---
## 2 · Análisis de diversidad entre modelos base
> La correlación de **errores** (no de predicciones) mide la complementariedad real.

In [ ]:
# ─── 2.1 CORRELACIÓN DE PREDICCIONES Y DE ERRORES ──────────────────────────
oof_df       = pd.DataFrame(meta_train, columns=BASE_NAMES)
residuals_df = pd.DataFrame(
    {name: y.values - meta_train[:, i] for i, name in enumerate(BASE_NAMES)}
)
corr     = oof_df.corr()
corr_res = residuals_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Diversidad entre modelos base', fontweight='bold', fontsize=13)

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.4f', cmap='RdYlGn',
            center=0.98, square=True, linewidths=0.5, ax=axes[0],
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 10})
axes[0].set_title('Correlación de predicciones OOF')

mask2 = np.triu(np.ones_like(corr_res, dtype=bool))
sns.heatmap(corr_res, mask=mask2, annot=True, fmt='.4f', cmap='RdYlGn_r',
            center=0.5, square=True, linewidths=0.5, ax=axes[1],
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 10})
axes[1].set_title('Correlación de ERRORES OOF\n(< 1 = complementarios)')

plt.tight_layout()
plt.savefig('sb_01_diversidad.png', bbox_inches='tight', dpi=120)
plt.show()

pairs = [(BASE_NAMES[i], BASE_NAMES[j], corr_res.iloc[i,j])
         for i in range(n_base) for j in range(i+1, n_base)]
pairs.sort(key=lambda x: x[2])
print('\n💡 Pares más complementarios (menor corr. de errores):')
for a, b, r in pairs[:4]:
    print(f'   {a:<15} ↔ {b:<15}  r_error = {r:.4f}')

In [ ]:
# ─── 2.2 ERROR POR QUINTIL DE PRECIO ───────────────────────────────────────
price_q = pd.qcut(y.values, q=5,
                  labels=['Q1\n(bajo)','Q2','Q3','Q4','Q5\n(alto)'])
fig, ax = plt.subplots(figsize=(13, 5))
x_pos   = np.arange(5)
width   = 0.13
offsets = np.linspace(-2.5*width, 2.5*width, n_base)
colors  = [COLOR_MAIN, COLOR_PURPLE, COLOR_ACCENT,
           COLOR_SUCCESS, COLOR_DANGER, '#8B5CF6']

for idx, (name, col) in enumerate(zip(BASE_NAMES, colors)):
    rmses = [rmsle(y.values[price_q==q], meta_train[price_q==q, idx])
             for q in price_q.cat.categories]
    ax.bar(x_pos + offsets[idx], rmses, width, label=name,
           color=col, alpha=0.82, edgecolor='white')

ax.set_xticks(x_pos)
ax.set_xticklabels(price_q.cat.categories)
ax.set_ylabel('RMSLE OOF por quintil')
ax.legend(fontsize=9)
ax.set_title('¿Dónde falla cada modelo? (un buen ensemble los complementa)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('sb_02_error_quintil.png', bbox_inches='tight', dpi=120)
plt.show()

---
## 3 · Blending — de simple a optimizado

In [ ]:
# ─── 3.1 PROMEDIO SIMPLE + SLSQP + DIFFERENTIAL EVOLUTION ─────────────────
def loss_blend(w, oof_matrix, y_true):
    w  = np.abs(w); w = w / w.sum()
    return rmsle(y_true, oof_matrix @ w)

# Promedio simple
rmsle_avg  = rmsle(y, meta_train.mean(axis=1))
test_avg   = meta_test.mean(axis=1)

# SLSQP (local, rápido)
res_slsqp = minimize(
    loss_blend, x0=np.ones(n_base)/n_base,
    args=(meta_train, y.values), method='SLSQP',
    bounds=[(0,1)]*n_base,
    constraints={'type':'eq','fun': lambda w: np.abs(w).sum()-1},
    options={'maxiter':5000,'ftol':1e-12}
)
w_slsqp   = np.abs(res_slsqp.x) / np.abs(res_slsqp.x).sum()
rmsle_slsqp = rmsle(y, meta_train @ w_slsqp)
test_slsqp  = meta_test @ w_slsqp

# Differential Evolution (global, más robusto)
res_de = differential_evolution(
    loss_blend, bounds=[(0,1)]*n_base,
    args=(meta_train, y.values),
    seed=42, maxiter=2000, tol=1e-12,
    popsize=20, mutation=(0.5,1.5), recombination=0.7
)
w_de     = np.abs(res_de.x) / np.abs(res_de.x).sum()
rmsle_de = rmsle(y, meta_train @ w_de)
test_de  = meta_test @ w_de

# Seleccionar el mejor
if rmsle_slsqp <= rmsle_de:
    w_blend_best, rmsle_blend, test_blend_best = w_slsqp, rmsle_slsqp, test_slsqp
    method_blend = 'SLSQP'
else:
    w_blend_best, rmsle_blend, test_blend_best = w_de, rmsle_de, test_de
    method_blend = 'Diff. Evolution'

print('─' * 52)
print(f'  Promedio simple        RMSLE = {rmsle_avg:.5f}')
print(f'  Blend SLSQP            RMSLE = {rmsle_slsqp:.5f}')
print(f'  Blend Diff. Evolution  RMSLE = {rmsle_de:.5f}')
print('─' * 52)
print(f'  ⭐ Mejor blend: {method_blend}  RMSLE = {rmsle_blend:.5f}')
print('\n  Pesos óptimos:')
for name, w in zip(BASE_NAMES, w_blend_best):
    bar = '█' * int(w * 50)
    print(f'    {name:<15} {w:.4f}  {bar}')

---
## 4 · Búsqueda del subconjunto óptimo de modelos
> No siempre más modelos = mejor ensemble. Probamos todos los subconjuntos posibles.

In [ ]:
# ─── 4.1 BÚSQUEDA EXHAUSTIVA DE SUBCONJUNTOS ────────────────────────────────
best_subset_score = 99.0
best_subset       = None
best_subset_w     = None
subset_log        = []

print(f'Evaluando {sum(len(list(combinations(range(n_base),r))) for r in range(2,n_base+1))} subconjuntos...')

for r in range(2, n_base + 1):
    for combo in combinations(range(n_base), r):
        idx     = list(combo)
        sub_oof = meta_train[:, idx]
        res = minimize(
            loss_blend, x0=np.ones(len(idx))/len(idx),
            args=(sub_oof, y.values), method='SLSQP',
            bounds=[(0,1)]*len(idx),
            constraints={'type':'eq','fun': lambda w: np.abs(w).sum()-1},
            options={'maxiter':1000,'ftol':1e-10}
        )
        w_sub = np.abs(res.x) / np.abs(res.x).sum()
        score = rmsle(y, sub_oof @ w_sub)
        subset_log.append({'modelos': [BASE_NAMES[i] for i in idx],
                           'n': len(idx), 'rmsle': score, 'w': w_sub, 'idx': idx})
        if score < best_subset_score:
            best_subset_score = score
            best_subset       = idx
            best_subset_w     = w_sub

test_subset = meta_test[:, best_subset] @ best_subset_w

print(f'\n✅ Búsqueda completada — {len(subset_log)} subconjuntos')
print(f'\n🏆 Mejor subconjunto:')
for name, w in zip([BASE_NAMES[i] for i in best_subset], best_subset_w):
    print(f'   {name:<15} {w:.4f}')
print(f'   RMSLE = {best_subset_score:.5f}')

# Top-10
subset_df = pd.DataFrame([
    {'Modelos': '+'.join(s['modelos']), 'N': s['n'], 'RMSLE': s['rmsle']}
    for s in subset_log
]).sort_values('RMSLE').head(10).reset_index(drop=True)

print('\nTop 10 subconjuntos:')
display(subset_df.style.background_gradient(cmap='RdYlGn_r', subset=['RMSLE'])
        .format({'RMSLE': '{:.5f}'}))

---
## 5 · Stacking Nivel 1 — Meta-learners comparados
> El stacking usa las OOF como features de un modelo de nivel superior.

In [ ]:
# ─── 5.1 COMPARAR META-LEARNERS (10-fold CV sobre meta-features) ───────────
meta_learners = {
    'Ridge α=0.01' : Ridge(alpha=0.01),
    'Ridge α=0.05' : Ridge(alpha=0.05),
    'Ridge α=0.1'  : Ridge(alpha=0.1),
    'Ridge α=0.5'  : Ridge(alpha=0.5),
    'Ridge α=1.0'  : Ridge(alpha=1.0),
    'Lasso α=0.001': Lasso(alpha=0.001, max_iter=10000),
    'Lasso α=0.005': Lasso(alpha=0.005, max_iter=10000),
    'ElasticNet'   : ElasticNet(alpha=0.005, l1_ratio=0.5, max_iter=10000),
}
meta_results = {}
kf_meta = KFold(n_splits=10, shuffle=True, random_state=42)

print('Meta-learners (10-fold CV):')
print('─' * 55)
for name, model in meta_learners.items():
    sc = np.sqrt(-cross_val_score(
        model, meta_train, y,
        scoring='neg_mean_squared_error', cv=kf_meta
    ))
    meta_results[name] = {'mean': sc.mean(), 'std': sc.std()}
    print(f'  {name:<20}  {sc.mean():.5f} ± {sc.std():.5f}')

best_meta_name  = min(meta_results, key=lambda k: meta_results[k]['mean'])
best_meta_score = meta_results[best_meta_name]['mean']
best_meta_model = meta_learners[best_meta_name]
print('─' * 55)
print(f'  ⭐ Mejor: {best_meta_name}  RMSLE={best_meta_score:.5f}')

best_meta_model.fit(meta_train, y)
test_stacking_l1  = best_meta_model.predict(meta_test)
rmsle_stacking_l1 = best_meta_score

if hasattr(best_meta_model, 'coef_'):
    print('\n  Coeficientes del meta-learner:')
    for nb_name, coef in zip(BASE_NAMES, best_meta_model.coef_):
        sign = '+' if coef >= 0 else '-'
        bar  = '█' * max(0, int(abs(coef)*300))
        print(f'    {nb_name:<15} {sign}{abs(coef):.5f}  {bar}')

---
## 6 · Stacking de 2 niveles — meta-modelo sobre meta-modelo

In [ ]:
# ─── 6.1 OOF ANIDADO DEL META-LEARNER (nivel 1 → nivel 2) ─────────────────
def stacking_oof(meta_tr, y_tr, meta_te, model, kf):
    """Genera OOF del meta-learner sin data leakage (K-Fold interno)."""
    oof        = np.zeros(len(y_tr))
    test_folds = []
    for tr_idx, val_idx in kf.split(meta_tr):
        m = type(model)(**model.get_params())
        m.fit(meta_tr[tr_idx], y_tr.iloc[tr_idx])
        oof[val_idx] = m.predict(meta_tr[val_idx])
        test_folds.append(m.predict(meta_te))
    return oof, np.mean(test_folds, axis=0)

print('Generando OOF del nivel 1 (10-fold interno)...')
oof_l1, test_l1 = stacking_oof(
    meta_train, y, meta_test, best_meta_model,
    KFold(n_splits=10, shuffle=True, random_state=42)
)
rmsle_oof_l1 = rmsle(y, oof_l1)
print(f'  OOF RMSLE L1: {rmsle_oof_l1:.5f}')

# Nivel 2: combinar OOF(L1) + blend como features
meta2_train = np.column_stack([oof_l1, meta_train @ w_blend_best])
meta2_test  = np.column_stack([test_l1, meta_test  @ w_blend_best])

scores_l2 = {}
for alpha in [0.01, 0.05, 0.1, 0.5]:
    sc = np.sqrt(-cross_val_score(
        Ridge(alpha=alpha), meta2_train, y,
        scoring='neg_mean_squared_error',
        cv=KFold(n_splits=10, shuffle=True, random_state=42)
    ))
    scores_l2[alpha] = sc.mean()
    print(f'  L2 Ridge α={alpha}:  RMSLE = {sc.mean():.5f}')

best_l2_alpha = min(scores_l2, key=scores_l2.get)
rmsle_l2      = scores_l2[best_l2_alpha]
model_l2      = Ridge(alpha=best_l2_alpha)
model_l2.fit(meta2_train, y)
test_l2 = model_l2.predict(meta2_test)
print(f'\n  ⭐ Stacking L2 (α={best_l2_alpha}):  RMSLE = {rmsle_l2:.5f}')

---
## 7 · Ensemble híbrido — Stacking + Blending combinados

In [ ]:
# ─── 7.1 COMBINAR CANDIDATOS FINALES ────────────────────────────────────────
candidates = {
    'Blend_best'    : (meta_train @ w_blend_best,       test_blend_best,  rmsle_blend),
    'Blend_subset'  : (meta_train[:,best_subset]@best_subset_w, test_subset, best_subset_score),
    'Stacking_L1'   : (oof_l1,                          test_l1,          rmsle_oof_l1),
    'Stacking_L2'   : (meta2_train[:,0],                 test_l2,          rmsle_l2),
}

cand_names  = list(candidates.keys())
cand_oof    = np.column_stack([v[0] for v in candidates.values()])
cand_test   = np.column_stack([v[1] for v in candidates.values()])

print('Candidatos finales:')
print('─' * 45)
for name, vals in candidates.items():
    print(f'  {name:<20}  RMSLE = {vals[2]:.5f}')
print('─' * 45)

res_hybrid  = minimize(
    loss_blend, x0=np.ones(len(cand_names))/len(cand_names),
    args=(cand_oof, y.values), method='SLSQP',
    bounds=[(0,1)]*len(cand_names),
    constraints={'type':'eq','fun': lambda w: np.abs(w).sum()-1},
    options={'maxiter':5000,'ftol':1e-12}
)
w_hybrid     = np.abs(res_hybrid.x) / np.abs(res_hybrid.x).sum()
rmsle_hybrid = rmsle(y, cand_oof @ w_hybrid)
test_hybrid  = cand_test @ w_hybrid

print(f'\n🏆 Ensemble HÍBRIDO  RMSLE = {rmsle_hybrid:.5f}')
print('   Pesos:')
for name, w in zip(cand_names, w_hybrid):
    bar = '█' * int(w * 50)
    print(f'   {name:<20} {w:.4f}  {bar}')

---
## 8 · Comparativa global — todos los ensambles

In [ ]:
# ─── 8.1 RANKING COMPLETO ───────────────────────────────────────────────────
all_results = {f'Base: {n}': s for n, s in base_scores.items()}
all_results.update({
    'Promedio simple'   : rmsle_avg,
    'Blend SLSQP'       : rmsle_slsqp,
    'Blend Diff. Evol.' : rmsle_de,
    'Blend subconjunto' : best_subset_score,
    'Stacking L1'       : rmsle_stacking_l1,
    'Stacking L2'       : rmsle_l2,
    'Híbrido (final)'   : rmsle_hybrid,
})

def tipo(name):
    if name.startswith('Base:'): return 'Modelo base'
    if 'Stack' in name:          return 'Stacking'
    if 'Híbrido' in name:        return 'Híbrido'
    return 'Blending'

ranking = pd.DataFrame(
    [{'Método': k, 'RMSLE_OOF': v, 'Tipo': tipo(k)} for k, v in all_results.items()]
).sort_values('RMSLE_OOF').reset_index(drop=True)

display(ranking.style
        .background_gradient(cmap='RdYlGn_r', subset=['RMSLE_OOF'])
        .format({'RMSLE_OOF': '{:.5f}'})
        .hide(axis='index'))

tipo_colors = {'Modelo base': '#94A3B8', 'Blending': COLOR_MAIN,
               'Stacking': COLOR_SUCCESS, 'Híbrido': COLOR_ACCENT}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Comparativa completa — Stacking vs Blending', fontweight='bold', fontsize=13)

ax = axes[0]
bar_colors = [tipo_colors[t] for t in ranking['Tipo']]
bars = ax.barh(ranking['Método'][::-1], ranking['RMSLE_OOF'][::-1],
               color=bar_colors[::-1], alpha=0.85, edgecolor='white')
for bar, val in zip(bars, ranking['RMSLE_OOF'][::-1]):
    ax.text(bar.get_width() + 0.0001, bar.get_y() + bar.get_height()/2,
            f'{val:.5f}', va='center', fontsize=8.5)
ax.set_xlabel('RMSLE (OOF)')
ax.set_title('Ranking — todos los métodos')
ax.set_xlim(0, ranking['RMSLE_OOF'].max() * 1.1)
legend_els = [Patch(facecolor=c, label=t) for t, c in tipo_colors.items()]
ax.legend(handles=legend_els, loc='lower right', fontsize=9)

ax = axes[1]
ens_df = ranking[ranking['Tipo'] != 'Modelo base'].copy()
ens_df['Mejora_bp'] = (best_base_score - ens_df['RMSLE_OOF']) * 10000
clrs2 = [tipo_colors[t] for t in ens_df['Tipo']]
bars2 = ax.bar(range(len(ens_df)), ens_df['Mejora_bp'],
               color=clrs2, alpha=0.85, edgecolor='white')
for bar, val in zip(bars2, ens_df['Mejora_bp']):
    color_t = COLOR_SUCCESS if val > 0 else COLOR_DANGER
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3 if val >= 0 else bar.get_height() - 1.5,
            f'{val:+.1f}', ha='center', fontsize=9, fontweight='bold', color=color_t)
ax.set_xticks(range(len(ens_df)))
ax.set_xticklabels(ens_df['Método'], rotation=20, ha='right', fontsize=9)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mejora vs mejor base (×10⁴ pts)')
ax.set_title(f'Mejora sobre {best_base_name} ({best_base_score:.5f})')

plt.tight_layout()
plt.savefig('sb_03_ranking_completo.png', bbox_inches='tight', dpi=120)
plt.show()

---
## 9 · Diagnóstico de residuos del ensemble ganador

In [ ]:
# ─── 9.1 RESIDUOS DEL MEJOR ENSEMBLE ───────────────────────────────────────
best_ens_name  = ranking.iloc[0]['Método']
best_ens_rmsle = ranking.iloc[0]['RMSLE_OOF']

oof_map = {
    'Promedio simple'   : meta_train.mean(axis=1),
    'Blend SLSQP'       : meta_train @ w_slsqp,
    'Blend Diff. Evol.' : meta_train @ w_de,
    'Blend subconjunto' : meta_train[:,best_subset] @ best_subset_w,
    'Stacking L1'       : oof_l1,
    'Stacking L2'       : meta2_train[:,0],
    'Híbrido (final)'   : cand_oof @ w_hybrid,
}
best_oof  = oof_map.get(best_ens_name, cand_oof @ w_hybrid)
residuals = y.values - best_oof
r2        = 1 - residuals.var() / y.var()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Diagnóstico de residuos — {best_ens_name}', fontweight='bold')

ax = axes[0,0]
ax.scatter(best_oof, residuals, alpha=0.3, s=12, color=COLOR_MAIN)
ax.axhline(0, color=COLOR_DANGER, linewidth=2, linestyle='--')
ax.axhline( 2*residuals.std(), color='gray', linewidth=1, linestyle=':', alpha=0.7)
ax.axhline(-2*residuals.std(), color='gray', linewidth=1, linestyle=':', alpha=0.7)
ax.set_xlabel('Predicción OOF'); ax.set_ylabel('Residuo')
ax.set_title('Residuos vs predicho')

ax = axes[0,1]
ax.hist(residuals, bins=40, color=COLOR_MAIN, alpha=0.8, edgecolor='white', density=True)
xr = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(xr, _norm.pdf(xr, residuals.mean(), residuals.std()),
        color=COLOR_DANGER, linewidth=2, label='Normal')
ax.set_xlabel('Residuo'); ax.set_ylabel('Densidad')
ax.set_title(f'Distribución  skew={_skew(residuals):.3f}')
ax.legend()

ax = axes[1,0]
_stats.probplot(residuals, dist='norm', plot=ax)
ax.get_lines()[0].set(color=COLOR_MAIN, markersize=3, alpha=0.4)
ax.get_lines()[1].set(color=COLOR_DANGER, linewidth=2)
ax.set_title('QQ Plot')

ax = axes[1,1]
ax.scatter(y.values, best_oof, alpha=0.3, s=12, color=COLOR_SUCCESS)
mn, mx = y.values.min(), y.values.max()
ax.plot([mn,mx],[mn,mx], color=COLOR_DANGER, linewidth=2, linestyle='--', label='y=x')
ax.set_xlabel('Real'); ax.set_ylabel('Predicho OOF')
ax.set_title(f'Real vs Predicho  R²={r2:.5f}')
ax.legend()

plt.tight_layout()
plt.savefig('sb_04_residuos.png', bbox_inches='tight', dpi=120)
plt.show()

print(f'📊 Residuos — {best_ens_name}')
print(f'   RMSLE: {best_ens_rmsle:.5f}  |  R²: {r2:.5f}')
print(f'   Media: {residuals.mean():.6f}  |  Std: {residuals.std():.6f}')
print(f'   |Max|: {np.abs(residuals).max():.6f}')

---
## 10 · Submission final

In [ ]:
# ─── 10.1 PREDICCIÓN DE TEST DEL ENSEMBLE GANADOR ──────────────────────────
test_map = {
    'Promedio simple'   : test_avg,
    'Blend SLSQP'       : test_slsqp,
    'Blend Diff. Evol.' : test_de,
    'Blend subconjunto' : test_subset,
    'Stacking L1'       : test_l1,
    'Stacking L2'       : test_l2,
    'Híbrido (final)'   : test_hybrid,
}

final_pred_log = test_map.get(best_ens_name, test_hybrid)
final_pred     = np.clip(np.expm1(final_pred_log), a_min=0, a_max=None)

submission = pd.DataFrame({'Id': test_ids.values, 'SalePrice': final_pred})
submission.to_csv('submission_stacking.csv', index=False)

assert len(submission) == len(test_ids)
assert submission['SalePrice'].isnull().sum() == 0
assert (submission['SalePrice'] > 0).all()

print(f'✅ submission_stacking.csv')
print(f'   Método  : {best_ens_name}')
print(f'   RMSLE   : {best_ens_rmsle:.5f}')
print(f'   Filas   : {len(submission)}')
print(f'   Rango   : ${final_pred.min():,.0f} — ${final_pred.max():,.0f}')
print(f'   Mediana : ${np.median(final_pred):,.0f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.expm1(y.values), bins=50, color=COLOR_MAIN,  alpha=0.5, edgecolor='white', label='Train real')
ax.hist(final_pred,          bins=50, color=COLOR_SUCCESS, alpha=0.6, edgecolor='white', label='Test predicho')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_xlabel('SalePrice'); ax.legend()
ax.set_title(f'Train vs Test — {best_ens_name}', fontweight='bold')
plt.tight_layout()
plt.savefig('sb_05_submission.png', bbox_inches='tight', dpi=120)
plt.show()

---
## 11 · Guardado de artefactos y resumen ejecutivo

In [ ]:
# ─── 11.1 GUARDAR ARTEFACTOS ────────────────────────────────────────────────
np.save('weights_blend_best.npy',  w_blend_best)
np.save('weights_blend_slsqp.npy', w_slsqp)
np.save('weights_blend_de.npy',    w_de)
np.save('weights_subset.npy',      best_subset_w)
np.save('weights_hybrid.npy',      w_hybrid)
np.save('oof_ensemble_final.npy',  best_oof)

joblib.dump(model_l2,        'model_stack_L2.pkl')
joblib.dump(best_meta_model, 'model_meta_L1.pkl')

with open('best_subset_indices.json', 'w') as f:
    json.dump({'indices': best_subset,
               'names':   [BASE_NAMES[i] for i in best_subset]}, f, indent=2)

ranking.to_csv('ensemble_ranking.csv', index=False)

print('✅ Artefactos guardados:')
files = {
    'weights_blend_best.npy'    : f'Pesos blend óptimo ({method_blend})',
    'weights_hybrid.npy'        : 'Pesos ensemble híbrido',
    'best_subset_indices.json'  : f'Subconjunto: {[BASE_NAMES[i] for i in best_subset]}',
    'model_meta_L1.pkl'         : f'Meta-learner L1 ({best_meta_name})',
    'model_stack_L2.pkl'        : f'Meta-learner L2 (α={best_l2_alpha})',
    'ensemble_ranking.csv'      : 'Ranking completo de ensambles',
    'oof_ensemble_final.npy'    : 'OOF del ensemble ganador',
    'submission_stacking.csv'   : 'Submission final Kaggle',
}
for fname, desc in files.items():
    if os.path.exists(fname):
        size = os.path.getsize(fname) / 1024
        print(f'   📄 {fname:<30} {size:6.1f} KB  — {desc}')

In [ ]:
# ─── 11.2 RESUMEN EJECUTIVO ─────────────────────────────────────────────────
mejora_bp = (best_base_score - best_ens_rmsle) * 10000

print()
print('═' * 68)
print('  RESUMEN EJECUTIVO — STACKING & BLENDING')
print('═' * 68)

print(f'\n📊 MODELOS BASE')
for name, score in sorted(base_scores.items(), key=lambda x: x[1]):
    print(f'   {name:<15} {score:.5f}')

print(f'\n🔀 ENSAMBLES EVALUADOS')
print(f'   Subconjuntos testados   : {len(subset_log)}')
print(f'   Meta-learners testados  : {len(meta_learners)}')
print(f'\n   Promedio simple         {rmsle_avg:.5f}')
print(f'   Blend SLSQP             {rmsle_slsqp:.5f}')
print(f'   Blend Diff. Evolution   {rmsle_de:.5f}')
print(f'   Blend subconjunto       {best_subset_score:.5f}  ← {[BASE_NAMES[i] for i in best_subset]}')
print(f'   Stacking L1             {rmsle_stacking_l1:.5f}  ← {best_meta_name}')
print(f'   Stacking L2             {rmsle_l2:.5f}')
print(f'   Híbrido                 {rmsle_hybrid:.5f}')

print(f'\n🏆 GANADOR: {best_ens_name}')
print(f'   OOF RMSLE   : {best_ens_rmsle:.5f}')
print(f'   vs {best_base_name}  : {best_base_score:.5f} → mejora {mejora_bp:+.1f} pts ×10⁴')
print(f'   R²          : {r2:.5f}')

print(f'\n📤 SUBMISSION: submission_stacking.csv')
print(f'   {len(submission)} filas  |  ${final_pred.min():,.0f} — ${final_pred.max():,.0f}')

print(f'''
💡 ESTRATEGIAS PARA MEJORAR AÚN MÁS
   1. Añadir CatBoost o SVR como 4to/5to modelo base → más diversidad
   2. Feature-weighted blending: pesos distintos por quintil de precio
   3. Bayesian Model Averaging como alternativa al blending
   4. Pseudo-labeling: usar predicciones de alta confianza del test
   5. Aumentar n_trials Optuna a 150+ para mejores modelos base
''')
print('═' * 68)